In [0]:
import json
from pyspark.sql.types import StructType, StructField, StringType, BooleanType
from pyspark.sql.functions import current_timestamp

# Read the JSON file
file_path = "/Workspace/Users/baodi.wilkinson.external@atradius.com/SAP_BO_Analytics/New Extraction/AD Users/All Users_10Jun26.json"

with open(file_path, "r") as f:
    users_data = json.loads(f.read())

print(f"Total users parsed: {len(users_data)}")

# Define schema matching All users.json structure
schema = StructType([
    StructField("id", StringType(), True),
    StructField("userId", StringType(), True),
    StructField("name", StringType(), True),
    StructField("mail", StringType(), True),
    StructField("businessUnit", StringType(), True),
    StructField("active", BooleanType(), True)
])

# Normalize records (fill missing optional fields with None)
rows = []
for user in users_data:
    rows.append((
        user.get("id"),
        user.get("userId"),
        user.get("name"),
        user.get("mail"),
        user.get("businessUnit"),
        user.get("active")
    ))

# Create Spark DataFrame and add ingestion timestamp
df_users = spark.createDataFrame(rows, schema=schema) \
    .withColumn("ingestion_ts", current_timestamp())
print(f"DataFrame row count: {df_users.count()}")
display(df_users.filter(df_users["userId"].isNotNull()))

In [0]:
# Write the parsed user data to Delta table
target_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.AD_User_basic"

# Collect userIds from target table
existing_user_ids = set(
    spark.table(target_table)
    .select("userId")
    .where("userId IS NOT NULL")
    .distinct()
    .rdd.flatMap(lambda x: x)
    .collect()
)

# Filter df_users for new userIds only
new_users_df = df_users.filter(~df_users["userId"].isin(existing_user_ids))

print(f"Existing users in target table: {len(existing_user_ids)}")
print(f"New users to add: {new_users_df.count()}")

# Append only new users
new_users_df.write.mode("append").option("mergeSchema", "true").saveAsTable(target_table)

print(f"Data loaded into {target_table}")
print(f"Row count: {spark.table(target_table).count()}")

In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.ad_user_basic
where to_date(Ingestion_ts) = to_date(current_date)

In [0]:
import json
from pyspark.sql.functions import current_timestamp, col

# Read the Full User Details JSON file
file_path = "/Workspace/Users/baodi.wilkinson.external@atradius.com/SAP_BO_Analytics/New Extraction/AD Users/Additional User Details_10Jun26"

with open(file_path, "r") as f:
    full_users_data = json.loads(f.read())

print(f"Total full user records parsed: {len(full_users_data)}")

# Create Spark DataFrame (infer schema from JSON) and add ingestion timestamp
df_full_users = spark.createDataFrame(full_users_data) \
    .withColumn("ingestion_ts", current_timestamp())

print(f"DataFrame row count: {df_full_users.count()}")
print(f"Columns: {df_full_users.columns}")
# display(df_full_users)

# Read existing AD_User_full table into dataframe
df_ad_user_full = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.ad_user_full")
print(f"\nExisting AD_User_full table row count: {df_ad_user_full.count()}")
print(f"Columns: {df_ad_user_full.columns}")
# display(df_ad_user_full)

# # Show differences in 'name' and 'manager' columns between the two dataframes
# df_diff = df_full_users.alias("new").join(
#     df_ad_user_full.alias("existing"),
#     on=col("new.uid") == col("existing.uid"),
#     how="inner"
# ).where(
#     (col("new.name") != col("existing.name")) |
#     (col("new.manager") != col("existing.manager")) |
#     (col("new.name").isNull() != col("existing.name").isNull()) |
#     (col("new.manager").isNull() != col("existing.manager").isNull())
# ).select(
#     col("new.uid").alias("uid"),
#     col("new.name").alias("new_name"),
#     col("existing.name").alias("existing_name"),
#     col("new.manager").alias("new_manager"),
#     col("existing.manager").alias("existing_manager")
# )

# print(f"\nRecords with differences in name or manager: {df_diff.count()}")
# # display(df_diff)

from pyspark.sql.functions import lit

# Ensure all required columns exist and are in the correct order
uc_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.ad_user_full"
uc_schema = spark.table(uc_table).schema
uc_columns = [f.name for f in uc_schema]

for c in uc_columns:
    if c not in df_full_users.columns:
        df_full_users = df_full_users.withColumn(c, lit(None).cast(uc_schema[c].dataType))

df_full_users = df_full_users.select(*uc_columns)
display(df_full_users)
# df_full_users.write.mode("append").option("mergeSchema", "true").saveAsTable(uc_table)
print(f"Data written to {uc_table}")
print(f"Row count: {spark.table(uc_table).count()}")


In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.ad_user_basic
where userId not in (select distinct user_ID from dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles)


In [0]:
from pyspark.sql.functions import coalesce, col, current_timestamp

# Target schema columns (excluding ingestion_ts which we regenerate)
target_columns = ["uid", "name", "businessUnit", "country", "countryCode", "manager", "timeZone", "level"]

# Ensure df_full_users has all required columns (add missing ones as null)
for c in target_columns:
    if c not in df_full_users.columns:
        df_full_users = df_full_users.withColumn(c, lit(None).cast("string"))
display(df_full_users)
# Ensure df_ad_user_full has all required columns (add missing ones as null)
for c in target_columns:
    if c not in df_ad_user_full.columns:
        df_ad_user_full = df_ad_user_full.withColumn(c, lit(None).cast("string"))

# Full outer join on uid - prioritize df_full_users values
df_merged = df_full_users.alias("new").join(
    df_ad_user_full.alias("existing"),
    on=col("new.uid") == col("existing.uid"),
    how="full_outer"
)

# For each column, take df_full_users ("new") value first, fall back to df_ad_user_full ("existing")
select_exprs = []
for c in target_columns:
    select_exprs.append(coalesce(col(f"new.{c}"), col(f"existing.{c}")).alias(c))

# Add fresh ingestion timestamp
df_merged_final = df_merged.select(*select_exprs).withColumn("ingestion_ts", current_timestamp())

print(f"df_full_users count: {df_full_users.count()}")
print(f"df_ad_user_full count: {df_ad_user_full.count()}")
print(f"Merged df count: {df_merged_final.count()}")
print(f"Schema: {df_merged_final.columns}")
display(df_merged_final)

In [0]:
%sql
Create or replace table dataplatform01_central_dev_catalog_01.custom_sap_bo.AD_User_profiles
as
select 
  AB1.userId AS user_ID, 
  AB1.name as full_name, 
  AB1.mail as email, 
  ab1.active as AD_status, 
  AB1.businessUnit as BU, 
  af1.countryCode as country, 
  af1.manager as line_manager, 
  af1.level as level, 
  af1.timeZone as time_zone, 
  af1.ingestion_ts as Ingestion_ts,
  CASE 
    WHEN UPPER(AB1.businessUnit) IN (
      'COMCE-GERMANY,AUSTRIA, SWITZERL,CEE','COMAS-ASIA','GLOBA-GLOBAL','COMNA-NORTH AMERICA',
      'COMNN-NETHERLANDS','COMON-OCEANIA','COMSE-BELGIUM, LUXEMBOURG','COMSPB-SPAIN, PORTUGAL, BRAZIL',
      'COMUI-UK & IRELAND','FRANC-FRANCE','ITALY-ITALY','CREDIT INSURANC'
    ) THEN 'Commercials'
    WHEN UPPER(AB1.businessUnit) IN (
      'RISK1-RS1-GERMANY, CENTRAL & EAST EUROPE','RISK2-RS2-BELGIUM, LUX, FRANCE & ITA',
      'RISK3-RS3-NLD, APAC & AIM','RISK4-RS4-NORTH EUROPE, CIS & ACS','RISK4-RS4-NORTH EUROPE, CIS & SP',
      'RISK5-RS5-NAFTA','RISK1-RS1-DEU, AUT AND CHE','RISK1-RS1-EUROPE, RUSSIA & CIS',
      'RISK2-RS2-NLD, BELUX, FRA, AFRICA & ISR','RISK3-RS3-ASIA AND OCEANIA',
      'RISK7-RS7-SPAIN, PORTUGAL, ANDORRA'
    ) THEN 'Risk'
    WHEN UPPER(AB1.businessUnit) IN (
      'FINPM-FINANCE PROGRAM MANAGEMENT','GFO-GROUP FINANCE OPERATIONS','GFC-GROUP FINANCE AND CONTROL',
      'COFIN-CORPORATE FINANCE & TAX','GROUP FINANCE','FINANCE','FINANCE & CONTROL'
    ) THEN 'Finance'
    ELSE '.Others'
  END AS BU_Group,
  CASE 
    WHEN AB1.userId IN (
      SELECT DISTINCT user_id FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.pbi_webi_basic
    ) THEN TRUE
    ELSE FALSE
  END AS is_BO_User
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.AD_User_basic as AB1
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.AD_User_full as AF1
on UPPER(tRIM(AF1.uid)) = upper(tRIM(AB1.userId))

-- select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.AD_User_profiles
-- where user_ID  like 'ESAFER8'

In [0]:
%sql
-- Compare full_name vs name from email
-- Match if every word from email name (normalized) is found as substring in full_name (normalized, concatenated)
-- Handles: accents, umlauts, apostrophes, multi-word surnames (De Bolle→Debolle), 
-- shortened names (Subhanullah contains Subhan), extra parts in full_name (LAMBODHARAN)
SELECT * FROM (
  SELECT 
    user_ID,
    full_name,
    INITCAP(REPLACE(SPLIT_PART(email, '@', 1), '.', ' ')) AS name_from_email,
    CASE 
      WHEN forall(
        filter(split(
          regexp_replace(
            regexp_replace(LOWER(REPLACE(SPLIT_PART(email, '@', 1), '.', ' ')), '\\bexternal\\b', ''),
          '[^a-z ]', ''), ' '), x -> trim(x) != '' AND length(trim(x)) > 1),
        word -> CONTAINS(
          regexp_replace(LOWER(
            translate(
              REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(full_name, 
                'ü', 'ue'), 'ö', 'oe'), 'ä', 'ae'), 'Ü', 'Ue'), 'Ö', 'Oe'), 'Ä', 'Ae'), 'ß', 'ss'),
              'àáâãåèéêëìíîïòóôõùúûýÿñçÀÁÂÃÅÈÉÊËÌÍÎÏÒÓÔÕÙÚÛÝŸÑÇøØ', 
              'aaaaaeeeeiiiioooouuuyyncAAAAAEEEEIIIIOOOOUUUYYNCoO')
          ), '[^a-z]', ''),
          trim(word)
        )
      )
      THEN 'Match'
      ELSE 'Mismatch'
    END AS comparison,
    email,
    BU
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles
)
WHERE comparison = 'Mismatch'
AND name_from_email IS NOT NULL
AND name_from_email LIKE '% %'
ORDER BY user_ID

In [0]:
%sql
-- Update full_name for the 161 genuine mismatches identified in cell 5
-- Applies the same filters: email not null, email has firstname.lastname format, and forall mismatch
update  dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles
SET full_name = INITCAP(REPLACE(SPLIT_PART(email, '@', 1), '.', ' '))
-- select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles
WHERE email IS NOT NULL
  AND INITCAP(REPLACE(SPLIT_PART(email, '@', 1), '.', ' ')) LIKE '% %'
  AND NOT forall(
    filter(split(
      regexp_replace(
        regexp_replace(LOWER(REPLACE(SPLIT_PART(email, '@', 1), '.', ' ')), '\\bexternal\\b', ''),
      '[^a-z ]', ''), ' '), x -> trim(x) != '' AND length(trim(x)) > 1),
    word -> CONTAINS(
      regexp_replace(LOWER(
        translate(
          REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(full_name, 
            'ü', 'ue'), 'ö', 'oe'), 'ä', 'ae'), 'Ü', 'Ue'), 'Ö', 'Oe'), 'Ä', 'Ae'), 'ß', 'ss'),
          'àáâãåèéêëìíîïòóôõùúûýÿñçÀÁÂÃÅÈÉÊËÌÍÎÏÒÓÔÕÙÚÛÝŸÑÇøØ', 
          'aaaaaeeeeiiiioooouuuyyncAAAAAEEEEIIIIOOOOUUUYYNCoO')
      ), '[^a-z]', ''),
      trim(word)
    )
  )

In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles
where upper(trim(user_ID)) in ('NLDCAP1')

In [0]:
import pandas as pd
import re

# Load user data
df_full = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles").toPandas()

# Normalize: collapse multiple spaces, strip, uppercase
def normalize(val):
    if pd.isna(val) or val is None:
        return None
    return re.sub(r'\s+', ' ', str(val).strip()).upper()

# Order-independent key: sort words alphabetically
def sorted_key(val):
    if val is None:
        return None
    return ' '.join(sorted(val.split()))

df_full['name_upper'] = df_full['full_name'].apply(normalize)
df_full['manager_upper'] = df_full['line_manager'].apply(normalize)
df_full['name_sorted'] = df_full['name_upper'].apply(sorted_key)

# Build primary lookup: exact name_upper -> record
lookup = df_full.drop_duplicates(subset='name_upper').set_index('name_upper')[['user_ID', 'full_name', 'level', 'BU', 'manager_upper']].to_dict('index')

# Build secondary lookup: sorted name -> record (for order-independent matching)
lookup_sorted = df_full.drop_duplicates(subset='name_sorted').set_index('name_sorted')[['user_ID', 'full_name', 'level', 'BU', 'manager_upper']].to_dict('index')

all_names = set(lookup.keys())
all_sorted = set(lookup_sorted.keys())
max_depth = 10

def find_manager(manager_upper):
    """Find manager record: try exact match first, then order-independent match."""
    if manager_upper in all_names:
        return lookup[manager_upper]
    # Try sorted key (handles name order differences)
    mgr_sorted = sorted_key(manager_upper)
    if mgr_sorted in all_sorted:
        return lookup_sorted[mgr_sorted]
    return None

display(df_full)
# For each user, trace UP to find their full reporting chain
rows = []
for _, user in df_full.iterrows():
    chain = []
    # Start with the user themselves
    chain.append({'uid': user['user_ID'], 'name': user['full_name'], 'level': user['level'], 'businessUnit': user['BU']})
    
    # Trace up through managers
    current_manager = user['manager_upper']
    visited = set()
    while current_manager and current_manager not in visited and len(chain) <= max_depth:
        visited.add(current_manager)
        mgr = find_manager(current_manager)
        if mgr:
            chain.append({'uid': mgr['user_ID'], 'name': mgr['full_name'], 'level': mgr['level'], 'businessUnit': mgr['BU']})
            current_manager = mgr['manager_upper']
        else:
            # Manager not in table - still include with name only
            chain.append({'uid': None, 'name': current_manager.title(), 'level': None, 'businessUnit': None})
            break
    
    # Reverse: chain[0] = top manager (L0), chain[-1] = current user
    chain.reverse()
    
    row = {}
    for i, person in enumerate(chain):
        row[f'L{i}_name'] = person['name']
        row[f'L{i}_uid'] = person['uid']
        row[f'L{i}_level'] = person['level']
        row[f'L{i}_businessUnit'] = person['businessUnit']
    rows.append(row)

# Build DataFrame
df_hierarchy = pd.DataFrame(rows)

# Sort columns by level number, then by attribute
level_cols = sorted(df_hierarchy.columns, key=lambda c: (int(c.split('_')[0][1:]), c.split('_', 1)[1]))
df_hierarchy = df_hierarchy[level_cols]

max_lvl = max(int(c.split('_')[0][1:]) for c in df_hierarchy.columns)
print(f"Reporting line table: {len(df_hierarchy)} rows, max depth: {max_lvl} levels")
print(f"Columns: {list(df_hierarchy.columns)}")

# Convert to Spark and display
df_reporting = spark.createDataFrame(df_hierarchy.astype(str).where(df_hierarchy.notna(), None))
display(df_reporting)

In [0]:
# Filter out rows where L0_uid or L0_businessUnit is null, then write hierarchy output to Delta table
target_table = "dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_Hierarchies"

df_reporting_filtered = df_reporting.filter("L0_uid IS NOT NULL AND L0_businessUnit IS NOT NULL")

df_reporting_filtered.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)

print(f"Data written to {target_table}")
print(f"Row count: {spark.table(target_table).count()}")

In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_Hierarchies

In [0]:
from pyspark.sql.functions import upper, trim, col
# Sort words in both line_manager and manager_name for order-independent comparison
from pyspark.sql.functions import split, array_sort, concat_ws

def sort_words(val):
    return ' '.join(sorted(val.strip().upper().split())) if val else None

# Set the email to look up
target_email = "Winfried.Gehrke@atradius.com"

# Load profiles
df_profiles = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles")

# Find the full_name for the given email
manager_row = df_profiles.filter(upper(trim(col("email"))) == target_email.strip().upper()).select("full_name", "user_ID").first()

if manager_row is None:
    print(f"No user found with email: {target_email}")
else:
    manager_name = manager_row["full_name"]
    manager_id = manager_row["user_ID"]
    print(f"Manager: {manager_name} ({manager_id})")
    
    # Count direct reports (where line_manager matches this person's full_name)

    sorted_manager = sort_words(manager_name)
    sorted_col = concat_ws(' ', array_sort(split(upper(trim(col("line_manager"))), '\\s+')))

    direct_reports = df_profiles.filter(
        sorted_col == sorted_manager
    )
    print(f"Direct reports: {direct_reports.count()}")
    
    # Count all reports using hierarchy table (all levels)
    # Each user has exactly one row; manager_id appears in that row at some L*_uid level.
    # If manager_id appears in a row, that person is in the manager's reporting chain.
    # Subtract 1 to exclude the manager's own row.
    df_hierarchy = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_Hierarchies")
    level_cols = [c for c in df_hierarchy.columns if c.endswith('_uid')]
    
    from functools import reduce
    from pyspark.sql.functions import lit
    condition = reduce(lambda a, b: a | b, [col(c) == lit(manager_id) for c in level_cols])
    total_count = df_hierarchy.filter(condition).count()
    print(f"All reports (all levels, excluding self): {total_count - 1 if total_count > 0 else 0}")
    
    print(f"\nDirect reports list:")
    display(direct_reports.select("user_ID", "full_name", "email", "level", "BU"))

In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.controller_team

In [0]:
%sql
ALTER TABLE dataplatform01_central_dev_catalog_01.
ADD COLUMN Is_Controller BOOLEAN AFTER BU;

ALTER TABLE dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.AD_User_basic
ADD COLUMN Is_Manager BOOLEAN AFTER Is_Controller;

ALTER TABLE dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.AD_User_basic
ADD COLUMN direct_report_count INT AFTER Is_Manager;

ALTER TABLE dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.AD_User_basic
ADD COLUMN all_report_count INT AFTER direct_report_count;
-- describe table dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.AD_User_basic
-- Set Is_Controller: TRUE if userId exists in controller_team
UPDATE dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.AD_User_basic AS ab
SET Is_Controller = EXISTS (
    SELECT 1 FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.controller_team AS ct
    WHERE UPPER(TRIM(ct.user_ID)) = UPPER(TRIM(ab.userId))
);

-- Set Is_Manager: TRUE if anyone in AD_User_profiles reports to this user
-- Uses sorted-word matching (order-independent) to handle name variations like "De Bolle Jan" vs "Jan De Bolle"
UPDATE dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.AD_User_basic AS ab
SET Is_Manager = EXISTS (
    SELECT 1 FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles AS ap
    WHERE concat_ws(' ', array_sort(split(UPPER(TRIM(ap.line_manager)), '\\s+')))
        = concat_ws(' ', array_sort(split(UPPER(TRIM(ab.name)), '\\s+')))
    AND UPPER(TRIM(ap.user_ID)) != UPPER(TRIM(ab.userId))
);

-- Set direct_report_count: number of people whose line_manager matches this user's name (sorted-word matching)
UPDATE dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.AD_User_basic AS ab
SET direct_report_count = (
    SELECT COUNT(*) FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles AS ap
    WHERE concat_ws(' ', array_sort(split(UPPER(TRIM(ap.line_manager)), '\\s+')))
        = concat_ws(' ', array_sort(split(UPPER(TRIM(ab.name)), '\\s+')))
    AND UPPER(TRIM(ap.user_ID)) != UPPER(TRIM(ab.userId))
);

-- Set all_report_count: number of people in hierarchy where this userId appears in any L*_uid column, minus 1 (self)
UPDATE dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.AD_User_basic AS ab
SET all_report_count = (
    SELECT GREATEST(COUNT(*) - 1, 0)
    FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_Hierarchies AS h
    WHERE h.L0_uid = ab.userId OR h.L1_uid = ab.userId OR h.L2_uid = ab.userId
       OR h.L3_uid = ab.userId OR h.L4_uid = ab.userId OR h.L5_uid = ab.userId
       OR h.L6_uid = ab.userId OR h.L7_uid = ab.userId
);

-- Verify
SELECT
  SUM(CASE WHEN Is_Controller THEN 1 ELSE 0 END) AS controllers,
  SUM(CASE WHEN Is_Manager THEN 1 ELSE 0 END) AS managers,
  SUM(CASE WHEN Is_Controller AND Is_Manager THEN 1 ELSE 0 END) AS both,
  SUM(direct_report_count) AS total_direct_reports,
  SUM(all_report_count) AS total_all_reports,
  COUNT(*) AS total
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.AD_User_basic

In [0]:
table = "dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles"
source = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.AD_User_basic"

# Add columns if not exists
existing_cols = {c.name for c in spark.table(table).schema}
new_cols = {"Is_Controller": "BOOLEAN", "Is_Manager": "BOOLEAN", "direct_report_count": "INT", "all_report_count": "INT"}

for col_name, col_type in new_cols.items():
    if col_name not in existing_cols:
        spark.sql(f"ALTER TABLE {table} ADD COLUMN {col_name} {col_type}")
        print(f"Added column: {col_name}")
    else:
        print(f"Column already exists: {col_name}")

# Update from AD_User_basic using user_ID = userId
print("\nUpdating ad_user_profiles from AD_User_basic...")
spark.sql(f"""
    MERGE INTO {table} AS ap
    USING {source} AS ab
    ON UPPER(TRIM(ap.user_ID)) = UPPER(TRIM(ab.userId))
    WHEN MATCHED THEN UPDATE SET
        ap.Is_Controller = ab.Is_Controller,
        ap.Is_Manager = ab.Is_Manager,
        ap.direct_report_count = ab.direct_report_count,
        ap.all_report_count = ab.all_report_count
""")

# Verify
print("\n=== ad_user_profiles VERIFICATION ===")
display(spark.sql(f"""
    SELECT
      SUM(CASE WHEN Is_Controller THEN 1 ELSE 0 END) AS controllers,
      SUM(CASE WHEN Is_Manager THEN 1 ELSE 0 END) AS managers,
      SUM(direct_report_count) AS total_direct_reports,
      SUM(all_report_count) AS total_all_reports,
      COUNT(*) AS total
    FROM {table}
"""))

# Sample: top managers by all_report_count
print("\n=== TOP 10 MANAGERS BY ALL REPORTS ===")
display(spark.sql(f"""
    SELECT user_ID, full_name, BU, Is_Controller, direct_report_count, all_report_count
    FROM {table}
    WHERE Is_Manager = TRUE
    ORDER BY all_report_count DESC
    LIMIT 10
"""))